In [23]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [24]:
#read dat
df = pd.read_csv('fallr.dat',
            header=None, sep='\s\s+', engine='python')

In [25]:
df

,0,1,2,3,4,5,6,7,8,9,10
0,OBS,ID,DSUM,EXER1,WEEK,ADL_R,SEX,NHOME,FYRPR,ADOR3,NFALL
1,1,1001,90,2,1,6,M,1,1,1,0
2,2,1001,90,2,2,6,M,1,1,1,0
3,3,1001,90,2,3,6,M,1,1,1,0
4,4,1001,90,2,4,6,M,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
2181,2181,2208,90,2,3,6,M,1,1,2,0
2182,2182,2208,90,2,4,6,M,1,1,2,0
2183,2183,2208,90,2,5,6,M,1,1,2,0
2184,2184,2208,90,2,6,6,M,1,1,2,0


In [26]:
#set row 1 as header
df.columns = df.iloc[0]
df = df[1:]

In [27]:
df.head()

,OBS,ID,DSUM,EXER1,WEEK,ADL_R,SEX,NHOME,FYRPR,ADOR3,NFALL
1,1,1001,90,2,1,6,M,1,1,1,0
2,2,1001,90,2,2,6,M,1,1,1,0
3,3,1001,90,2,3,6,M,1,1,1,0
4,4,1001,90,2,4,6,M,1,1,1,0
5,5,1001,90,2,5,6,M,1,1,1,0


In [28]:
df = df[["SEX", "NFALL"]] # select sex nfall column

In [30]:
df

,SEX,NFALL
1,M,0
2,M,0
3,M,0
4,M,0
5,M,0
...,...,...
2181,M,0
2182,M,0
2183,M,0
2184,M,0


In [31]:
df = df.dropna(how='any', axis = 0) # drop na

In [32]:
df

,SEX,NFALL
1,M,0
2,M,0
3,M,0
4,M,0
5,M,0
...,...,...
2181,M,0
2182,M,0
2183,M,0
2184,M,0


In [8]:
#df['SEX'] = df['SEX'].astype(int)
df['NFALL'] = df['NFALL'].astype(int) # change data type

In [9]:
df.loc[df["NFALL"] >= 1, "NFALL"] = 1 # Nfall > 1 之值=1 ，count value

In [10]:
df['NFALL'].value_counts()

0    2089
1      95
Name: NFALL, dtype: int64

In [11]:
female = df[df['SEX'] == 'F']
female.value_counts()

SEX  NFALL
F    0        1317
     1          48
dtype: int64

In [12]:
male = df[df['SEX'] == 'M']
male.value_counts()

SEX  NFALL
M    0        772
     1         47
dtype: int64

In [13]:
df.groupby('SEX')['NFALL'].value_counts() / df.groupby('SEX')['NFALL'].count() # compute probability 

SEX  NFALL
F    0        0.964835
     1        0.035165
M    0        0.942613
     1        0.057387
Name: NFALL, dtype: float64

In [15]:
formula = "NFALL ~ SEX " #set function

In [16]:
model = smf.glm(formula = formula, data=df, family=sm.families.Binomial()) #  #GLM logistic model


In [17]:
result = model.fit()

In [18]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  NFALL   No. Observations:                 2184
Model:                            GLM   Df Residuals:                     2182
Model Family:                Binomial   Df Model:                            1
Link Function:                  logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -387.78
Date:                Wed, 21 Dec 2022   Deviance:                       775.57
Time:                        19:11:43   Pearson chi2:                 2.18e+03
No. Iterations:                     6                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.3119      0.147    -22.539      0.000      -3.600      -3.024
SEX[T.M]       0.5131      0.210      2.441      0.015       0.101       0.925
==============================================================================
"""

In [19]:
import numpy as np
# ... Define and fit model
odds_ratios = pd.DataFrame(
    {
        "OR": result.params,
        "Lower CI": result.conf_int()[0],
        "Upper CI": result.conf_int()[1],
    }
)
odds_ratios = np.exp(odds_ratios)
print(odds_ratios)


                 OR  Lower CI  Upper CI
Intercept  0.036446  0.027326  0.048611
SEX[T.M]   1.670418  1.106476  2.521786


In [20]:
np.exp(-3.3119+0.5131) / (1+np.exp(-3.3119+0.5131)) # male fall probability

0.057389056093863335

In [21]:
np.exp(-3.3119) / (1+np.exp(-3.3119)) # female fall probability

0.035165197888182356